# YOLO26x Pyro-SDIS resume (budget9) | Kaggle GPU

- Settings: GPU T4 x2, Internet on (KHÔNG chọn P100 — torch 2.10+cu128 của Kaggle đã bỏ sm_60)
- Input 1 (dataset): zip `pyro-sdis-yolo.zip`, slug `DATASET_SLUG`
- Input 2 (checkpoint): file `last_resume.pt`, slug `CHECKPOINT_SLUG`
- Train tiếp epoch 19-20/20 (epoch 18 đã xong trên Modal 2026-07-15); mọi hyperparam (seed/imgsz/batch/patience/augmentation) lấy nguyên từ checkpoint, không set lại

In [ ]:
from pathlib import Path

DATASET_SLUG = 'pyro-sdis-yolo'
CHECKPOINT_SLUG = 'yolo26x-pyro-sdis-budget9-ckpt'
RUN_NAME = 'yolo26x_pyro_sdis_budget9'

INPUT_ROOT = Path('/kaggle/input')
if not (INPUT_ROOT / DATASET_SLUG).is_dir():
    INPUT_ROOT = next(p for p in (INPUT_ROOT / 'datasets').iterdir() if (p / DATASET_SLUG).is_dir())
WORK_ROOT = Path('/kaggle/working')
RUN_DIR = WORK_ROOT / 'runs' / RUN_NAME

In [ ]:
import subprocess

subprocess.run(['nvidia-smi'], check=True)
subprocess.run([
    'pip', 'install', '-q',
    'torch==2.4.1', 'torchvision==0.19.1',
    '--index-url', 'https://download.pytorch.org/whl/cu118',
], check=True)
subprocess.run(['pip', 'install', '-q', '--no-deps', 'ultralytics==8.4.90', 'ultralytics-thop'], check=True)
subprocess.run([
    'pip', 'install', '-q',
    'numpy', 'matplotlib', 'opencv-python', 'pillow', 'pyyaml', 'requests', 'psutil', 'polars', 'nvidia-ml-py',
], check=True)

import torch
import ultralytics

print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'ultralytics': ultralytics.__version__})
assert torch.cuda.is_available(), 'CUDA không khả dụng'
print('arch list:', torch.cuda.get_arch_list())
assert any('60' in arch for arch in torch.cuda.get_arch_list()), f'torch build này vẫn không có sm_60: {torch.cuda.get_arch_list()}'
probe = torch.nn.Conv2d(3, 8, 3).cuda()
probe(torch.randn(1, 3, 32, 32, device='cuda'))
print('CUDA probe OK')
capability = torch.cuda.get_device_capability(0)
assert capability >= (7, 0), f'{torch.cuda.get_device_name(0)} sm_{capability[0]}{capability[1]} không được torch {torch.__version__} hỗ trợ — đổi Accelerator sang GPU T4 x2'


In [ ]:
import shutil

import yaml


def find_under_input(filename, slug):
    matches = sorted(p for p in INPUT_ROOT.rglob(filename) if slug in p.parts)
    assert matches, f'không tìm thấy {filename} cho slug {slug} dưới {INPUT_ROOT}'
    return matches[0]


manifest_path = find_under_input('dataset.yaml', DATASET_SLUG)
data_root = manifest_path.parent
print('dataset root:', data_root)
assert (data_root / 'images' / 'train').is_dir(), f'dataset không đúng cấu trúc: {data_root}'

data_yaml_path = WORK_ROOT / 'dataset.yaml'
data_yaml_path.write_text(
    yaml.safe_dump(
        {'path': str(data_root), 'train': 'images/train', 'val': 'images/val', 'names': {0: 'smoke'}},
        sort_keys=False,
    ),
    encoding='utf-8',
)

checkpoint_source = find_under_input('last_resume.pt', CHECKPOINT_SLUG)
checkpoint_target = RUN_DIR / 'weights' / 'last_resume.pt'
checkpoint_target.parent.mkdir(parents=True, exist_ok=True)
if not checkpoint_target.exists():
    shutil.copy2(checkpoint_source, checkpoint_target)

_ckpt = torch.load(checkpoint_target, map_location='cpu', weights_only=False)
print('resume from completed epoch', _ckpt['epoch'] + 1, '-> target total epochs', _ckpt['train_args']['epochs'])
del _ckpt

In [ ]:
from ultralytics import YOLO
from ultralytics.models.yolo.detect.train import DetectionTrainer


def make_resume_trainer(data_path, project, name):
    class ResumeTrainer(DetectionTrainer):
        def _rebind_runtime_paths(self):
            self.args.data = str(data_path)
            self.args.project = str(project)
            self.args.name = name
            self.save_dir = Path(project) / name
            self.args.save_dir = str(self.save_dir)

        def check_resume(self, overrides):
            super().check_resume(overrides)
            self._rebind_runtime_paths()

        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self._rebind_runtime_paths()

    return ResumeTrainer


def on_model_save(trainer):
    last_path = RUN_DIR / 'weights' / 'last.pt'
    resume_path = RUN_DIR / 'weights' / 'last_resume.pt'
    checkpoint = torch.load(last_path, map_location='cpu', weights_only=False)
    checkpoint['scheduler_state'] = trainer.scheduler.state_dict()
    checkpoint['scaler'] = trainer.scaler.state_dict()
    torch.save(checkpoint, resume_path)


model = YOLO(str(checkpoint_target))
model.add_callback('on_model_save', on_model_save)
model.train(
    resume=str(checkpoint_target),
    data=str(data_yaml_path),
    trainer=make_resume_trainer(data_yaml_path, RUN_DIR.parent, RUN_NAME),
)

In [ ]:
for path in sorted((RUN_DIR / 'weights').glob('*')):
    print(path, path.stat().st_size)
assert (RUN_DIR / 'weights' / 'last_resume.pt').is_file()